In [1]:
!pip install transformers torch pandas -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Posts with usable text: {df['has_text'].sum()}")
print(f"\nText source breakdown:")
print(df['text_source'].value_counts())

Loaded: 905 rows, 22 columns
Posts with usable text: 797

Text source breakdown:
text_source
caption_only          652
caption+transcript    143
none                  108
transcript_only         2
Name: count, dtype: int64


In [4]:
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    truncation=True
)

# Quick test
print(emotion_model("I'm so excited for the Super Bowl!"))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12626.20it/s]


[[{'label': 'joy', 'score': 0.9424753189086914}, {'label': 'surprise', 'score': 0.04108182713389397}, {'label': 'neutral', 'score': 0.010181290097534657}, {'label': 'anger', 'score': 0.0025986728724092245}, {'label': 'sadness', 'score': 0.0020127270836383104}, {'label': 'fear', 'score': 0.0012048218632116914}, {'label': 'disgust', 'score': 0.00044535406050272286}]]


In [5]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:2000] for t in texts]   # safety cap — model truncates at 512 tokens anyway

print(f"Running emotion model on {len(texts)} posts...")
results = emotion_model(texts, batch_size=16)
print(f"Got {len(results)} results")

Running emotion model on 797 posts...
Got 797 results


In [6]:
emotion_labels = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
emo_cols = [f"emo_{c}" for c in emotion_labels]

# Drop existing emotion columns if they exist (safe re-run)
df = df.drop(columns=[c for c in emo_cols + ["dominant_emotion", "dominant_emotion_score"] if c in df.columns])

# Build emotion dataframe
rows = [{item["label"]: item["score"] for item in res} for res in results]
emo_df = pd.DataFrame(rows)[emotion_labels]
emo_df.columns = emo_cols
emo_df.index = df.index[df["has_text"]]
df = df.join(emo_df)

# Compute dominant emotion (only on rows with text)
df["dominant_emotion"] = None
df["dominant_emotion_score"] = None
df.loc[df["has_text"], "dominant_emotion"] = (
    df.loc[df["has_text"], emo_cols].idxmax(axis=1).str.replace("emo_", "")
)
df.loc[df["has_text"], "dominant_emotion_score"] = df.loc[df["has_text"], emo_cols].max(axis=1)

print(df[["student_id", "text_source", "dominant_emotion", "dominant_emotion_score"]].head())

  student_id         text_source dominant_emotion dominant_emotion_score
0        1_A  caption+transcript            anger               0.863525
1        1_A  caption+transcript         surprise               0.626648
2        1_A        caption_only          neutral                0.64811
3        2_A        caption_only          neutral               0.914405
4        2_A        caption_only          neutral               0.913741


In [7]:
print("=== Overall dominant emotion counts ===")
print(df["dominant_emotion"].value_counts(dropna=False))
print()
print("=== Mean scores across all usable posts ===")
print(df[emo_cols].mean().sort_values(ascending=False))
print()
print("=== Per-student dominant emotion breakdown ===")
print(pd.crosstab(df["student_id"], df["dominant_emotion"]))

=== Overall dominant emotion counts ===
dominant_emotion
neutral     281
joy         182
fear        146
None        108
surprise     83
anger        46
sadness      41
disgust      18
Name: count, dtype: int64

=== Mean scores across all usable posts ===
emo_neutral     0.301784
emo_joy         0.219502
emo_fear        0.170971
emo_surprise    0.119665
emo_anger       0.085234
emo_sadness     0.070694
emo_disgust     0.032151
dtype: float64

=== Per-student dominant emotion breakdown ===
dominant_emotion  anger  disgust  fear  joy  neutral  sadness  surprise
student_id                                                             
10_A                  0        1    13   11       14        9         9
11_A                  3        8    23   48       88        7         9
12_A                 14        0    20   18        7        3         7
1_A                   2        2     2    3       35        2        11
2_A                   1        1     8   21       17        3        11
3_

In [8]:
print("=== Dominant emotion by text source ===")
print(pd.crosstab(df["text_source"], df["dominant_emotion"]))
print()
print("=== Average dominant emotion confidence by text source ===")
print(df.groupby("text_source")["dominant_emotion_score"].agg(["mean", "median", "count"]))

=== Dominant emotion by text source ===
dominant_emotion    anger  disgust  fear  joy  neutral  sadness  surprise
text_source                                                              
caption+transcript     17        7    34   17       44        6        18
caption_only           29       11   112  165      235       35        65
transcript_only         0        0     0    0        2        0         0

=== Average dominant emotion confidence by text source ===
                        mean    median  count
text_source                                  
caption+transcript  0.662656   0.68017    143
caption_only         0.70102  0.726264    652
none                     NaN       NaN      0
transcript_only     0.581849  0.581849      2


In [9]:
# Look at top anger and fear posts from caption+transcript group
for emo in ["anger", "fear", "disgust"]:
    subset = df[(df["dominant_emotion"] == emo) & (df["text_source"] == "caption+transcript")]
    print(f"\n=== {emo.upper()} from caption+transcript ({len(subset)} posts) ===")
    if len(subset) == 0:
        continue
    top = subset.nlargest(3, f"emo_{emo}")
    for _, r in top.iterrows():
        text = r['text_for_analysis'][:400].replace('\n', ' | ')
        print(f"\n  [{r[f'emo_{emo}']:.2f}] [{r['student_id']}]")
        print(f"  {text}...")


=== ANGER from caption+transcript (17 posts) ===

  [0.99] [12_A]
  precise and deadly. || Book 5: Lightning #anime #viralvideo #avatarthelastairbender #fyp??viral #corecore |  | Lightning is a pure expression of firebending without aggression. It is not fueled by rage or emotion the way other firebending is. Some call lightning the cold-blooded fire. It is precise and deadly....

  [0.92] [12_A]
  #ethanwinters ; this game really stayed with me. after finishing the ending of the Shadows of Rose DLC from Resident Evil Village, I couldnt stop thinking about the story for a long time. Seeing how much Ethan was willing to do for his daughter was such a powerful moment. This game will definitely stay as one of the most memorable ones for me #residentevilvillage #rosewinters #fup #shadowsofrose | ...

  [0.91] [6_A]
  brb going absolutely feral at chanel #chanel #matthieublazy |  | Okay, so I'm at Chanel looking at the new Mathieu Blassé collection, and I just decided between these pair o

In [10]:
# Check how many posts share transcripts (indicating data errors)
with_tr = df[df["has_transcript"]].copy()
transcript_counts = with_tr["Full Transcription"].value_counts()
duplicated = transcript_counts[transcript_counts > 1]

print(f"Total posts with transcripts: {len(with_tr)}")
print(f"Unique transcripts: {with_tr['Full Transcription'].nunique()}")
print(f"Transcripts appearing more than once: {len(duplicated)}")
print(f"\nMost duplicated transcripts:")
print(duplicated.head(10))

Total posts with transcripts: 145
Unique transcripts: 139
Transcripts appearing more than once: 5

Most duplicated transcripts:
Full Transcription
And kudos to our general manager to get Gunther outta here. (crowd booing) Oh!\nIs that Lee? IT is, it's Dragon Lee. Dragon Lee attacking Gunther on the night that his friend AJ Styles, his former tag team partner is to be celebrated. Dragon Lee's had enough and he takes it upon himself to attack Gunther. Dragon Lee tired of the disrespect being displayed towards styles by Gunther. Good on Dragon Lee. (group shouting) And Dragon Lee not taking up. (group shouting) Get him off! Get him off! Get him off! (group shouting) Out! Get him out! Are you happy? And now heading up top. Uso Splash into the cover. One, two, three. Jay Uso is going the chamber. Here is your winner, one half of the World Tag Team Champions, main event Jay Uso. It is very clear to me that you are the champion that you are because of what you had to go through. But now I nee

In [11]:
output_path = "../Outputs/emotion_distilroberta_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/emotion_distilroberta_905.csv


# Emotion Model 1: DistilRoBERTa — Conclusion

**Model:** `j-hartmann/emotion-english-distilroberta-base`
**Output labels:** 7 emotions — anger, disgust, fear, joy, neutral, sadness, surprise
**Input used:** caption + transcript (combined) where available, caption only otherwise
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

We ran every post through a model that reads text and decides what emotion is in it. The model gives a score from 0 to 1 for each of the 7 emotions. The one with the highest score becomes the "dominant emotion" for that post.

## What we found overall

Most posts came out as either neutral (35%) or joy (23%). Fear was third at 18%. The negative emotions (anger, sadness, disgust) were all small. This roughly matches what you'd expect from a normal social media feed.

## The interesting comparison: caption only vs caption + transcript

We had 143 posts where students provided both a caption AND a video transcript. The other 652 posts only had captions. We compared the two groups.

**What changed when transcripts were added:**
- Neutral and joy dropped
- Anger went up 3x (from 4% to 12%)
- Fear went up (from 17% to 24%)
- Disgust roughly doubled

At first glance this looks like "transcripts help the model see negative emotions the captions hid." But when we actually looked at the posts, we found the opposite story.

## The problem we found

The model is keyword-reactive. It looks at the text, sees emotion-related words, and fires the emotion label whether or not the post is actually about that emotion.

Three real examples from the top "anger" posts:

- An Avatar cartoon clip with the transcript "lightning is a pure expression of firebending without aggression, it is NOT fueled by rage" was tagged 99% anger. The transcript literally says "not rage" and the model still called it anger.
- A wholesome post about a Resident Evil video game's emotional story was tagged 92% anger. The caption says "this game really stayed with me." Not angry.
- A light Chanel shopping vlog was tagged 91% anger because the student said "going feral" (which is slang for excited, not angry).

The same pattern shows up for fear. Batman comic posts, music videos, friendly Spider-Man crossover content all get tagged as fear because they mention villains, shadows, or ghosts.

## What this actually means

Adding transcripts didn't make the model more accurate. It just gave the model more keywords to misread. Longer text means more chances to find emotion-related words, which means more false positives on the negative emotions.

The model is not understanding what posts are about. It's pattern-matching on words.

## Data quality note

While checking the data, we found 6 posts in the dataset where the wrong transcript was attached. For example, a post about Trump's tariffs has a wedding dialogue transcript. This is a small data issue (~4% of posts with transcripts) but worth flagging for the next round of data cleanup.

## What this output gives us

A clean CSV with all 7 emotion scores for every post, plus the dominant emotion label. This is useful as a baseline, but the per-post labels should be treated as noisy. Patterns at the student level (which students tend to consume which emotions) are probably more trustworthy than any individual post's label.

## Bottom line

The pipeline works. The model gives outputs. But the outputs are unreliable on entertainment content and any post that mentions emotion words ironically or in passing. This is exactly why we're running multiple models tonight: to see whether other transformer models behave the same way, or whether some of them handle context better.